In [10]:
import os
import re
import glob
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [11]:
# ==========================================================
# CONFIGURATION
# ==========================================================
INPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/Turning_angle"
FIG_BASE_DIR = r"./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/"
SMOOTH_DIR = r"./../../dataFolders/MuscaChasingBeads/xyz_smooth"
os.makedirs(FIG_BASE_DIR, exist_ok=True)

In [15]:

# Font size settings
TITLE_SIZE = 24
LABEL_SIZE = 20
TICK_SIZE  = 16

csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_TURNING_ANGLE.csv")))

# ==========================================================
# PLOTTING
# ==========================================================
for path in csv_files:
    base = os.path.splitext(os.path.basename(path))[0]

    # 1. Extract Trial Name
    match = re.search(r"(Trial\d+_\d+rpm)", base)
    trial_name = match.group(1) if match else "Trial_Data"

    # 2. Create Trial-Specific Sub-folder
    trial_save_dir = os.path.join(FIG_BASE_DIR, trial_name)
    os.makedirs(trial_save_dir, exist_ok=True)

    print(f"Generating plots for trial: {trial_name}")

    # Load turning angle data
    df = pd.read_csv(path)
    t  = df["frame"].values

    # --- Helper function for 2D plots ---
    def save_styled_plot(filename, title, ylabel, data_col, color, add_hline=False):
        plt.figure(figsize=(14, 7))
        plt.plot(t, df[data_col], color=color, linewidth=2.5)
        if add_hline:
            plt.axhline(0, color='black', linewidth=2, linestyle='-')
        plt.title(f"{title}: {trial_name}", fontsize=TITLE_SIZE, fontweight='bold', pad=20)
        plt.xlabel("Frame", fontsize=LABEL_SIZE)
        plt.ylabel(ylabel, fontsize=LABEL_SIZE)
        plt.xticks(fontsize=TICK_SIZE)
        plt.yticks(fontsize=TICK_SIZE)
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.tight_layout()
        plt.savefig(os.path.join(trial_save_dir, f"{base}_{filename}.png"), dpi=300)
        plt.close()

    # Plot 1: 3D Turning Angle
    save_styled_plot("1_3D_Angle",  "3D Turning Angle",        "Angle (°)",      "turning_angle_deg",       "darkgreen")
    # Plot 2: Turning Velocity
    save_styled_plot("2_Velocity",  "Turning Velocity",         "Velocity (°/s)", "turning_velocity_deg_s",  "firebrick")
    # Plot 3: XY Component (Yaw)
    save_styled_plot("3_XY_Yaw",   "XY Turning Angle (Yaw)",   "Angle (°)",      "turning_angle_xy_deg",    "blue",   add_hline=True)
    # Plot 4: YZ Component (Pitch)
    save_styled_plot("4_YZ_Pitch", "YZ Turning Angle (Pitch)",  "Angle (°)",      "turning_angle_yz_deg",    "orange", add_hline=True)

    # --- 3D Interactive Trajectory Plot ---
    # Load matching SMOOTH file for head + center positions
    smooth_path = glob.glob(os.path.join(SMOOTH_DIR, f"{trial_name}*_SMOOTH.csv"))
    if smooth_path:
        df_smooth = pd.read_csv(smooth_path[0])
        head   = df_smooth[["pt2_X", "pt2_Y", "pt2_Z"]].values
        center = df_smooth[["center_X", "center_Y", "center_Z"]].values

        turn_3d = df["turning_angle_deg"].values[:]   

        x = head[1:, 0]
        y = head[1:, 1]
        z = head[1:, 2]

        fig = go.Figure(data=go.Scatter3d(
            x=x, y=y, z=z,
            mode="lines+markers",
            marker=dict(
                size=3,
                color=turn_3d,
                colorscale="Turbo",
                colorbar=dict(title="Turning Angle (°)"),
                showscale=True,
            ),
            line=dict(
                color=turn_3d,
                colorscale="Turbo",
                width=3,
            ),
            hovertemplate=(
                "Frame: %{text}<br>"
                "X: %{x:.4f}<br>"
                "Y: %{y:.4f}<br>"
                "Z: %{z:.4f}<br>"
                "Turning Angle: %{marker.color:.2f}°"
                "<extra></extra>"
            ),
            text=df_smooth["frame"].values[1:],
        ))

        fig.update_layout(
            title=f"{trial_name} — 3D Trajectory (coloured by Turning Angle)",
            scene=dict(
                xaxis_title="X",
                yaxis_title="Y",
                zaxis_title="Z",
            ),
            width=1000,
            height=700,
        )

        out_html = os.path.join(trial_save_dir, f"{base}_5_3D_trajectory.html")
        fig.write_html(out_html)
        print(f"  Saved 3D plot: {out_html}")
    else:
        print(f"  WARNING: No SMOOTH file found for {trial_name} — skipping 3D plot")

print(f"\nAll plots saved in: {FIG_BASE_DIR}")

Generating plots for trial: Trial2_180rpm
  Saved 3D plot: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/Trial2_180rpm\Trial2_180rpmxyzpts_TURNING_ANGLE_5_3D_trajectory.html
Generating plots for trial: Trial2_200rpm
  Saved 3D plot: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/Trial2_200rpm\Trial2_200rpmxyzpts_TURNING_ANGLE_5_3D_trajectory.html
Generating plots for trial: Trial3_180rpm
  Saved 3D plot: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/Trial3_180rpm\Trial3_180rpmxyzpts_TURNING_ANGLE_5_3D_trajectory.html
Generating plots for trial: Trial4_400rpm
  Saved 3D plot: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/Trial4_400rpm\Trial4_400rpmxyzpts_TURNING_ANGLE_5_3D_trajectory.html
Generating plots for trial: Trial5_180rpm
  Saved 3D plot: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/Trial5_180rpm\Trial5_180rpmxyzpts_TURNING_ANGLE_5_3D_trajectory.html
Generating plots for trial: Trial5_400rp